# 🍎 Apple Inc. (AAPL) — Financial Intelligence Report
### Deep-Dive Analysis: FY2022 – FY2025
---
**Tools:** Python · Pandas · NumPy · SQLite · Plotly · Matplotlib  
**Data Source:** Yahoo Finance via `yfinance`  
**Scope:** Quarterly income statement, balance sheet, cash flow statement + stock price  

> *This notebook was built as a capstone project for*  
> *Accounting Data Analytics with Python — University of Illinois Urbana-Champaign (Coursera)*

---
## 📦 Section 1 — Environment Setup
> Install and import all required libraries.

In [ ]:
# Kaggle already has most packages; reinstall yfinance to ensure latest version
import subprocess
subprocess.run(['pip', 'install', 'yfinance', '--upgrade', '-q'])
subprocess.run(['pip', 'install', 'plotly', '--upgrade', '-q'])
print('✅ Packages ready')

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import sqlite3
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.ticker as mticker
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# ── Global style constants ────────────────────────────────────
GOLD, BLUE, GREEN, RED, PURPLE = '#C9A84C', '#4A9EFF', '#2ECC71', '#E05252', '#A78BFA'
TEAL, ORANGE = '#1ABC9C', '#F39C12'
BG, SURFACE, BORDER = '#0A0C10', '#111318', '#1E2330'

plt.rcParams.update({
    'axes.facecolor': '#0E1117',
    'figure.facecolor': BG,
    'text.color': '#E8EAF0',
    'axes.labelcolor': '#9AA0B0',
    'xtick.color': '#9AA0B0',
    'ytick.color': '#9AA0B0',
    'axes.edgecolor': BORDER,
    'grid.color': BORDER,
    'axes.spines.top': False,
    'axes.spines.right': False,
})

print('✅ All libraries loaded')
print(f'📅 Analysis run: {datetime.now().strftime("%Y-%m-%d %H:%M")}')

---
## 🗂️ Section 2 — Data Collection
> Fetch Apple's quarterly financials and daily stock price from Yahoo Finance (2022–2025).

In [ ]:
TICKER = 'AAPL'
START  = '2022-01-01'
END    = '2025-12-31'

aapl = yf.Ticker(TICKER)
print(f'📥 Fetching data for: {TICKER}')
info = aapl.info
print(f'   Company  : {info.get("longName", "Apple Inc.")}')
print(f'   Sector   : {info.get("sector", "Technology")}')
print(f'   Industry : {info.get("industry", "Consumer Electronics")}')
print(f'   Market Cap: ${info.get("marketCap", 0)/1e12:.2f}T')
print(f'   Exchange : {info.get("exchange", "NASDAQ")}')

In [ ]:
# ── Stock Price History ───────────────────────────────────────
df_price = aapl.history(start=START, end=END)
df_price = df_price.reset_index()
df_price['Date'] = pd.to_datetime(df_price['Date']).dt.tz_localize(None)
df_price['Ticker'] = TICKER

# Add technical indicators
df_price['MA_50']  = df_price['Close'].rolling(50).mean()
df_price['MA_200'] = df_price['Close'].rolling(200).mean()
df_price['Daily_Return'] = df_price['Close'].pct_change() * 100
df_price['Cumulative_Return'] = (1 + df_price['Close'].pct_change()).cumprod() - 1

print(f'✅ Stock price: {len(df_price)} trading days ({START} ~ {END})')
print(f'   Price range: ${df_price["Close"].min():.2f} ~ ${df_price["Close"].max():.2f}')
print(f'   Latest close: ${df_price["Close"].iloc[-1]:.2f}')
df_price.tail(3)

In [ ]:
# ── Quarterly Income Statement ────────────────────────────────
raw_inc = aapl.quarterly_financials.T.copy()
raw_inc.index = pd.to_datetime(raw_inc.index)
raw_inc = raw_inc.sort_index()
raw_inc = raw_inc[(raw_inc.index >= START) & (raw_inc.index <= END)]

want_inc = ['Total Revenue','Gross Profit','Operating Income','Net Income','EBITDA']
inc_cols = [c for c in want_inc if c in raw_inc.columns]
df_income = raw_inc[inc_cols].copy()
for col in inc_cols:
    df_income[col] = pd.to_numeric(df_income[col], errors='coerce') / 1e9

df_income['Quarter'] = df_income.index.to_period('Q').astype(str)
df_income['Gross_Margin'] = (df_income['Gross Profit']     / df_income['Total Revenue'] * 100).round(2)
df_income['Op_Margin']    = (df_income['Operating Income'] / df_income['Total Revenue'] * 100).round(2)
df_income['Net_Margin']   = (df_income['Net Income']       / df_income['Total Revenue'] * 100).round(2)
df_income['Revenue_YoY']  = df_income['Total Revenue'].pct_change(4).mul(100).round(2)
df_income['Revenue_QoQ']  = df_income['Total Revenue'].pct_change(1).mul(100).round(2)
if 'EBITDA' in df_income.columns:
    df_income['EBITDA_Margin'] = (df_income['EBITDA'] / df_income['Total Revenue'] * 100).round(2)

print(f'✅ Income statement: {len(df_income)} quarters')
df_income[['Quarter'] + inc_cols].tail(8)

In [ ]:
# ── Quarterly Balance Sheet ───────────────────────────────────
raw_bal = aapl.quarterly_balance_sheet.T.copy()
raw_bal.index = pd.to_datetime(raw_bal.index)
raw_bal = raw_bal.sort_index()
raw_bal = raw_bal[(raw_bal.index >= START) & (raw_bal.index <= END)]

want_bal = ['Total Assets','Total Liabilities Net Minority Interest',
            'Stockholders Equity','Total Debt','Cash And Cash Equivalents',
            'Current Assets','Current Liabilities']
bal_cols = [c for c in want_bal if c in raw_bal.columns]
df_balance = raw_bal[bal_cols].copy()
for col in bal_cols:
    df_balance[col] = pd.to_numeric(df_balance[col], errors='coerce') / 1e9

df_balance['Quarter'] = df_balance.index.to_period('Q').astype(str)
if 'Total Liabilities Net Minority Interest' in df_balance.columns:
    df_balance['Debt_Ratio'] = (df_balance['Total Liabilities Net Minority Interest'] / df_balance['Total Assets'] * 100).round(2)
if 'Current Assets' in df_balance.columns and 'Current Liabilities' in df_balance.columns:
    df_balance['Current_Ratio'] = (df_balance['Current Assets'] / df_balance['Current Liabilities']).round(3)
if 'Stockholders Equity' in df_balance.columns:
    df_balance['D_to_E'] = (df_balance.get('Total Debt', np.nan) / df_balance['Stockholders Equity']).round(3)

print(f'✅ Balance sheet: {len(df_balance)} quarters')
df_balance[['Quarter','Total Assets','Total Debt','Stockholders Equity','Debt_Ratio']].tail(6)

In [ ]:
# ── Quarterly Cash Flow ───────────────────────────────────────
raw_cf = aapl.quarterly_cashflow.T.copy()
raw_cf.index = pd.to_datetime(raw_cf.index)
raw_cf = raw_cf.sort_index()
raw_cf = raw_cf[(raw_cf.index >= START) & (raw_cf.index <= END)]

want_cf = ['Operating Cash Flow','Capital Expenditure','Free Cash Flow',
           'Repurchase Of Capital Stock','Cash Dividends Paid']
cf_cols = [c for c in want_cf if c in raw_cf.columns]
df_cf = raw_cf[cf_cols].copy()
for col in cf_cols:
    df_cf[col] = pd.to_numeric(df_cf[col], errors='coerce') / 1e9

df_cf['Quarter'] = df_cf.index.to_period('Q').astype(str)
if 'Free Cash Flow' not in df_cf.columns:
    if 'Operating Cash Flow' in df_cf.columns and 'Capital Expenditure' in df_cf.columns:
        df_cf['Free Cash Flow'] = df_cf['Operating Cash Flow'] + df_cf['Capital Expenditure']
if 'Operating Cash Flow' in df_cf.columns:
    df_cf['FCF_Margin'] = (df_cf['Free Cash Flow'] / df_income['Total Revenue'].reindex(df_cf.index) * 100).round(2)

print(f'✅ Cash flow: {len(df_cf)} quarters')
df_cf[['Quarter','Operating Cash Flow','Capital Expenditure','Free Cash Flow']].tail(8)

---
## 🗄️ Section 3 — SQLite Database
> Store all cleaned data in a structured SQLite database. Demonstrate SQL queries for financial reporting.

In [ ]:
DB_PATH = 'apple_financial.db'
conn = sqlite3.connect(DB_PATH)

df_price.to_sql('stock_price',   conn, if_exists='replace', index=False)
df_income.to_sql('income_stmt',  conn, if_exists='replace', index=True,  index_label='Date')
df_balance.to_sql('balance_sheet',conn,if_exists='replace', index=True,  index_label='Date')
df_cf.to_sql('cash_flow',        conn, if_exists='replace', index=True,  index_label='Date')

print(f'✅ Database created: {DB_PATH}')
for t in ['stock_price','income_stmt','balance_sheet','cash_flow']:
    n = pd.read_sql(f'SELECT COUNT(*) AS n FROM {t}', conn).iloc[0,0]
    print(f'   {t:18s}: {n} rows')
conn.close()

In [ ]:
# ── SQL Query 1: Quarterly Revenue & Margin Summary ──────────
conn = sqlite3.connect(DB_PATH)
q1 = '''
SELECT
    Quarter,
    ROUND("Total Revenue", 2)   AS Revenue_Bn,
    ROUND(Gross_Margin, 1)      AS Gross_Margin_Pct,
    ROUND(Op_Margin, 1)         AS Op_Margin_Pct,
    ROUND(Net_Margin, 1)        AS Net_Margin_Pct,
    ROUND(Revenue_YoY, 1)       AS YoY_Growth_Pct
FROM income_stmt
ORDER BY Date DESC
LIMIT 12
'''
result1 = pd.read_sql(q1, conn)
print('📊 Apple Quarterly Performance Summary:')
print(result1.to_string(index=False))
conn.close()

In [ ]:
# ── SQL Query 2: Cash Flow Health ─────────────────────────────
conn = sqlite3.connect(DB_PATH)
q2 = '''
SELECT
    Quarter,
    ROUND("Operating Cash Flow", 2) AS OCF_Bn,
    ROUND("Free Cash Flow", 2)      AS FCF_Bn,
    ROUND("Capital Expenditure", 2) AS Capex_Bn
FROM cash_flow
ORDER BY Date DESC
LIMIT 12
'''
result2 = pd.read_sql(q2, conn)
print('💰 Apple Cash Flow Summary:')
print(result2.to_string(index=False))
conn.close()

In [ ]:
# ── SQL Query 3: Balance Sheet Strength ───────────────────────
conn = sqlite3.connect(DB_PATH)
q3 = '''
SELECT
    Quarter,
    ROUND("Total Assets", 2)        AS Assets_Bn,
    ROUND("Total Debt", 2)          AS Debt_Bn,
    ROUND("Stockholders Equity", 2) AS Equity_Bn,
    ROUND(Debt_Ratio, 1)            AS Debt_Ratio_Pct
FROM balance_sheet
ORDER BY Date DESC
LIMIT 8
'''
result3 = pd.read_sql(q3, conn)
print('🏦 Apple Balance Sheet Summary:')
print(result3.to_string(index=False))
conn.close()

---
## 📊 Section 4 — Financial Analysis
> Compute key financial ratios and metrics used in investment banking and FP&A.

In [ ]:
# ── Key Financial Ratios ──────────────────────────────────────
df_kpi = df_income[['Quarter','Total Revenue','Gross_Margin','Op_Margin','Net_Margin','Revenue_YoY','Revenue_QoQ']].copy()

# Merge with balance sheet for ROE / ROA
df_kpi = df_kpi.join(df_balance[['Stockholders Equity','Total Assets','Debt_Ratio','Current_Ratio']], how='left')
df_kpi = df_kpi.join(df_cf[['Free Cash Flow','Operating Cash Flow']], how='left')

df_kpi['ROE'] = (df_income['Net Income'] / df_balance['Stockholders Equity'].reindex(df_income.index) * 100).round(2)
df_kpi['ROA'] = (df_income['Net Income'] / df_balance['Total Assets'].reindex(df_income.index) * 100).round(2)
df_kpi['FCF_Yield'] = (df_cf['Free Cash Flow'] / df_income['Total Revenue'] * 100).round(2)

print('='*75)
print('  APPLE INC. — KEY FINANCIAL RATIOS (2022–2025)')
print('='*75)
display_cols = ['Quarter','Gross_Margin','Op_Margin','Net_Margin','ROE','ROA','FCF_Yield','Debt_Ratio']
display_cols = [c for c in display_cols if c in df_kpi.columns]
print(df_kpi[display_cols].tail(12).to_string(index=False))

In [ ]:
# ── Annual Aggregation (FY view) ─────────────────────────────
df_income_copy = df_income.copy()
df_income_copy['FY'] = df_income_copy.index.year

annual = df_income_copy.groupby('FY').agg(
    Annual_Revenue=('Total Revenue','sum'),
    Annual_NetIncome=('Net Income','sum'),
    Avg_Gross_Margin=('Gross_Margin','mean'),
    Avg_Net_Margin=('Net_Margin','mean'),
    Avg_Op_Margin=('Op_Margin','mean'),
).round(2)
annual['Revenue_Growth_YoY'] = annual['Annual_Revenue'].pct_change().mul(100).round(2)

print('='*70)
print('  APPLE INC. — ANNUAL SUMMARY (USD Bn)')
print('='*70)
print(annual.to_string())

In [ ]:
# ── Descriptive Statistics ────────────────────────────────────
print('='*60)
print('  APPLE — MARGIN STATISTICS (2022–2025)')
print('='*60)
margin_stats = df_income[['Gross_Margin','Op_Margin','Net_Margin']].describe().round(2)
print(margin_stats.to_string())

print('\n'+'='*60)
print('  APPLE — PRICE STATISTICS (2022–2025)')
print('='*60)
price_stats = df_price[['Close','Volume','Daily_Return']].describe().round(3)
print(price_stats.to_string())

---
## 📈 Section 5 — Visualization
### Part A — [Plotly] Interactive Investment Dashboard

In [ ]:
Q  = df_income['Quarter'].tolist()
ai = df_income
ab = df_balance
ac = df_cf
ah = df_price
h6 = ah[ah['Date'] >= ah['Date'].max() - pd.Timedelta(days=180)]

fig = make_subplots(
    rows=4, cols=3,
    subplot_titles=[
        'QUARTERLY REVENUE (USD Bn)', 'GROSS / OP / NET MARGIN (%)', 'OPERATING INCOME (USD Bn)',
        'FREE CASH FLOW (USD Bn)',    'REVENUE YoY GROWTH (%)',       'EBITDA (USD Bn)',
        'TOTAL ASSETS vs EQUITY (USD Bn)', 'DEBT RATIO (%)',          'STOCK PRICE 2022–2025 (USD)',
        'OHLC CANDLESTICK — LAST 6 MONTHS', '', ''
    ],
    specs=[
        [{'type':'bar'},     {'type':'scatter'}, {'type':'bar'}],
        [{'type':'bar'},     {'type':'bar'},     {'type':'bar'}],
        [{'type':'bar'},     {'type':'scatter'}, {'type':'scatter'}],
        [{'colspan':3,'type':'candlestick'}, None, None],
    ],
    vertical_spacing=0.09, horizontal_spacing=0.06,
    row_heights=[0.22, 0.22, 0.22, 0.34],
)

# R1C1 Revenue bars
rev = ai['Total Revenue'].values
fig.add_trace(go.Bar(
    x=Q, y=rev,
    marker_color=[GOLD if v == max(rev) else BLUE for v in rev],
    marker_line_width=0, showlegend=False,
    text=[f'{v:.1f}' for v in rev], textposition='outside',
    textfont=dict(size=9, color='white', family='Courier New')
), row=1, col=1)

# R1C2 Margin lines
for col, nm, c, d in [('Gross_Margin','Gross%',GREEN,'solid'),('Op_Margin','Op%',GOLD,'dash'),('Net_Margin','Net%',PURPLE,'dot')]:
    if col in ai.columns:
        fig.add_trace(go.Scatter(
            x=Q, y=ai[col].values, name=nm, mode='lines+markers',
            line=dict(color=c, width=2.5, dash=d), marker=dict(size=6)
        ), row=1, col=2)

# R1C3 Operating Income
if 'Operating Income' in ai.columns:
    op = ai['Operating Income'].values
    fig.add_trace(go.Bar(
        x=Q, y=op, marker_color=[GREEN if v>=0 else RED for v in op],
        marker_line_width=0, showlegend=False,
        text=[f'{v:.1f}' for v in op], textposition='outside',
        textfont=dict(size=9, color='white', family='Courier New')
    ), row=1, col=3)

# R2C1 FCF
if 'Free Cash Flow' in ac.columns:
    fcf = ac['Free Cash Flow'].values
    fig.add_trace(go.Bar(
        x=ac['Quarter'].tolist(), y=fcf,
        marker_color=[GREEN if v>=0 else RED for v in fcf],
        marker_line_width=0, showlegend=False,
        text=[f'{v:.1f}' for v in fcf], textposition='outside',
        textfont=dict(size=9, color='white', family='Courier New')
    ), row=2, col=1)

# R2C2 YoY Growth
if 'Revenue_YoY' in ai.columns:
    yoy = ai['Revenue_YoY'].fillna(0).values
    fig.add_trace(go.Bar(
        x=Q, y=yoy,
        marker_color=[GREEN if v>=0 else RED for v in yoy],
        marker_line_width=0, showlegend=False,
        text=[f'{v:.1f}%' for v in yoy], textposition='outside',
        textfont=dict(size=9, color='white', family='Courier New')
    ), row=2, col=2)

# R2C3 EBITDA
if 'EBITDA' in ai.columns:
    eb = ai['EBITDA'].dropna().values
    qeb = Q[:len(eb)]
    fig.add_trace(go.Bar(
        x=qeb, y=eb,
        marker_color=[GOLD if v==max(eb) else '#2A4A7F' for v in eb],
        marker_line_width=0, showlegend=False,
        text=[f'{v:.1f}' for v in eb], textposition='outside',
        textfont=dict(size=9, color='white', family='Courier New')
    ), row=2, col=3)

# R3C1 Assets vs Equity
if 'Total Assets' in ab.columns:
    fig.add_trace(go.Bar(
        x=ab['Quarter'].tolist(), y=ab['Total Assets'].values,
        name='Assets', marker_color='rgba(74,158,255,0.6)', marker_line_width=0
    ), row=3, col=1)
if 'Stockholders Equity' in ab.columns:
    fig.add_trace(go.Bar(
        x=ab['Quarter'].tolist(), y=ab['Stockholders Equity'].values,
        name='Equity', marker_color='rgba(201,168,76,0.8)', marker_line_width=0
    ), row=3, col=1)

# R3C2 Debt Ratio
if 'Debt_Ratio' in ab.columns:
    fig.add_trace(go.Scatter(
        x=ab['Quarter'].tolist(), y=ab['Debt_Ratio'].values,
        name='Debt%', mode='lines+markers+text',
        line=dict(color=RED, width=2.5), marker=dict(size=8),
        fill='tozeroy', fillcolor='rgba(224,82,82,0.08)',
        text=[f'{v:.0f}%' for v in ab['Debt_Ratio'].values],
        textposition='top center', textfont=dict(size=8, family='Courier New')
    ), row=3, col=2)

# R3C3 Stock Price
fig.add_trace(go.Scatter(
    x=ah['Date'], y=ah['Close'],
    name='AAPL', mode='lines',
    line=dict(color=GOLD, width=2),
    fill='tozeroy', fillcolor='rgba(201,168,76,0.06)'
), row=3, col=3)

# R4 OHLC Candlestick
fig.add_trace(go.Candlestick(
    x=h6['Date'], open=h6['Open'], high=h6['High'],
    low=h6['Low'], close=h6['Close'], name='OHLC',
    increasing=dict(line=dict(color=GREEN, width=1), fillcolor='rgba(46,204,113,0.6)'),
    decreasing=dict(line=dict(color=RED,   width=1), fillcolor='rgba(224,82,82,0.6)'),
), row=4, col=1)

cp = ah['Close'].iloc[-1]
fig.update_layout(
    title=dict(
        text=(f"<b>APPLE INC.  ·  AAPL  ·  NASDAQ</b>"
              f"<br><span style='font-size:12px;color:#5A6070;font-family:Courier New'>"
              f"FY2022–2025  |  Latest Close: ${cp:,.2f}  |  Currency: USD</span>"),
        font=dict(size=22, color=GOLD, family='Georgia, serif'),
        x=0.01, xanchor='left', y=0.997
    ),
    height=1500, template='plotly_dark',
    paper_bgcolor=BG, plot_bgcolor=SURFACE,
    font=dict(color='#E8EAF0', family='Courier New', size=10),
    barmode='group',
    legend=dict(orientation='h', y=1.005, x=0.5, xanchor='center',
                bgcolor='rgba(0,0,0,0)', font=dict(size=9, color='#5A6070')),
    margin=dict(t=95, b=50, l=55, r=40),
    xaxis_rangeslider_visible=False,
)
for ann in fig.layout.annotations:
    ann.font.size=9; ann.font.color='#5A6070'; ann.font.family='Courier New'

fig.show()
print('✅ Plotly interactive dashboard rendered')

### Part B — [Matplotlib] Static Investment Report

In [ ]:
fig = plt.figure(figsize=(22, 16))
fig.patch.set_facecolor(BG)
gs = gridspec.GridSpec(3, 3, figure=fig, hspace=0.50, wspace=0.35)

fig.text(0.02, 0.975, 'APPLE INC.  ·  AAPL  ·  NASDAQ',
         fontsize=22, color=GOLD, fontweight='bold', fontfamily='serif')
fig.text(0.02, 0.955, f'Financial Intelligence Report  |  FY2022–2025  |  All figures in USD Bn unless stated',
         fontsize=10, color='#5A6070', fontfamily='monospace')

def sax(ax, title):
    ax.set_facecolor('#111318')
    ax.set_title(title, color='#9AA0B0', fontsize=10, pad=9, fontfamily='monospace', fontweight='bold')
    ax.tick_params(colors='#5A6070', labelsize=8)
    ax.grid(axis='y', color=BORDER, alpha=0.9, linewidth=0.6)
    ax.spines[:].set_edgecolor(BORDER)

Q_list = df_income['Quarter'].tolist()
x_idx  = range(len(Q_list))

# ── Panel 1: Revenue ──────────────────────────────────────────
ax1 = fig.add_subplot(gs[0, 0]); sax(ax1, 'QUARTERLY REVENUE (USD Bn)')
bar_colors = [GOLD if v == max(df_income['Total Revenue'].values) else '#2A4A7F' for v in df_income['Total Revenue'].values]
bars = ax1.bar(x_idx, df_income['Total Revenue'].values, color=bar_colors, alpha=0.9, width=0.65)
for bar, val in zip(bars, df_income['Total Revenue'].values):
    ax1.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.5, f'{val:.0f}',
             ha='center', va='bottom', fontsize=7, color='#9AA0B0', fontfamily='monospace')
ax1.set_xticks(x_idx); ax1.set_xticklabels(Q_list, rotation=45, ha='right', fontsize=7)

# ── Panel 2: Margin Trend ─────────────────────────────────────
ax2 = fig.add_subplot(gs[0, 1]); sax(ax2, 'MARGIN TREND (%)')
for col, lbl, c, ls in [('Gross_Margin','Gross%',GREEN,'-'),('Op_Margin','Op%',GOLD,'--'),('Net_Margin','Net%',PURPLE,':')]:
    if col in df_income.columns:
        ax2.plot(x_idx, df_income[col].values, color=c, lw=2.5, ls=ls, marker='o', ms=5, label=lbl)
ax2.set_xticks(x_idx); ax2.set_xticklabels(Q_list, rotation=45, ha='right', fontsize=7)
ax2.legend(fontsize=8, facecolor='#111318', edgecolor=BORDER, labelcolor='#E8EAF0')
ax2.grid(axis='both', alpha=0.2)

# ── Panel 3: FCF ──────────────────────────────────────────────
ax3 = fig.add_subplot(gs[0, 2]); sax(ax3, 'FREE CASH FLOW (USD Bn)')
if 'Free Cash Flow' in df_cf.columns:
    cf_q = df_cf['Quarter'].tolist()
    cf_v = df_cf['Free Cash Flow'].values
    cf_colors = [GREEN if v >= 0 else RED for v in cf_v]
    ax3.bar(range(len(cf_q)), cf_v, color=cf_colors, alpha=0.85, width=0.65)
    for i, val in enumerate(cf_v):
        ax3.text(i, val + (0.3 if val >= 0 else -0.8), f'{val:.1f}',
                 ha='center', fontsize=7, color='#9AA0B0', fontfamily='monospace')
    ax3.set_xticks(range(len(cf_q))); ax3.set_xticklabels(cf_q, rotation=45, ha='right', fontsize=7)
ax3.axhline(0, color='#5A6070', lw=0.8)

# ── Panel 4: YoY Growth ───────────────────────────────────────
ax4 = fig.add_subplot(gs[1, 0]); sax(ax4, 'REVENUE YoY GROWTH (%)')
if 'Revenue_YoY' in df_income.columns:
    yoy = df_income['Revenue_YoY'].fillna(0).values
    ax4.bar(x_idx, yoy, color=[GREEN if v >= 0 else RED for v in yoy], alpha=0.85, width=0.65)
    for i, val in enumerate(yoy):
        ax4.text(i, val + (0.2 if val >= 0 else -1.5), f'{val:.1f}%',
                 ha='center', fontsize=7, color='#9AA0B0', fontfamily='monospace')
    ax4.set_xticks(x_idx); ax4.set_xticklabels(Q_list, rotation=45, ha='right', fontsize=7)
ax4.axhline(0, color='#5A6070', lw=0.8)

# ── Panel 5: Assets vs Equity ─────────────────────────────────
ax5 = fig.add_subplot(gs[1, 1]); sax(ax5, 'ASSETS vs EQUITY (USD Bn)')
xb = np.arange(len(df_balance))
if 'Total Assets' in df_balance.columns:
    ax5.bar(xb-0.2, df_balance['Total Assets'].values, 0.38, color=BLUE, alpha=0.55, label='Assets')
if 'Stockholders Equity' in df_balance.columns:
    ax5.bar(xb+0.2, df_balance['Stockholders Equity'].values, 0.38, color=GOLD, alpha=0.85, label='Equity')
ax5.set_xticks(xb); ax5.set_xticklabels(df_balance['Quarter'].tolist(), rotation=45, ha='right', fontsize=7)
ax5.legend(fontsize=8, facecolor='#111318', edgecolor=BORDER, labelcolor='#E8EAF0')

# ── Panel 6: ROE & ROA ────────────────────────────────────────
ax6 = fig.add_subplot(gs[1, 2]); sax(ax6, 'ROE vs ROA (%)')
if 'ROE' in df_kpi.columns and 'ROA' in df_kpi.columns:
    kq = df_kpi['Quarter'].tolist()
    ax6.plot(range(len(kq)), df_kpi['ROE'].values, color=GOLD, lw=2.5, marker='o', ms=5, label='ROE')
    ax6.plot(range(len(kq)), df_kpi['ROA'].values, color=TEAL, lw=2.5, marker='s', ms=5, ls='--', label='ROA')
    ax6.set_xticks(range(len(kq))); ax6.set_xticklabels(kq, rotation=45, ha='right', fontsize=7)
    ax6.legend(fontsize=8, facecolor='#111318', edgecolor=BORDER, labelcolor='#E8EAF0')
    ax6.grid(axis='both', alpha=0.2)

# ── Panel 7: Stock Price with MAs (full width) ────────────────
ax7 = fig.add_subplot(gs[2, :]); sax(ax7, 'STOCK PRICE WITH MA50 & MA200 (USD)  |  2022–2025')
ax7.fill_between(df_price['Date'], df_price['Close'], color=GOLD, alpha=0.08)
ax7.plot(df_price['Date'], df_price['Close'], color=GOLD, lw=1.8, label='AAPL Close', alpha=0.95)
ax7.plot(df_price['Date'], df_price['MA_50'],  color=BLUE,  lw=1.5, ls='--', label='MA50',  alpha=0.85)
ax7.plot(df_price['Date'], df_price['MA_200'], color=RED,   lw=1.5, ls=':',  label='MA200', alpha=0.85)
ax7.legend(fontsize=9, facecolor='#111318', edgecolor=BORDER, labelcolor='#E8EAF0', loc='upper left')
ax7.grid(axis='both', alpha=0.15)
ax7.yaxis.set_major_formatter(mticker.FormatStrFormatter('$%.0f'))

plt.savefig('apple_report.png', dpi=150, bbox_inches='tight', facecolor=BG, edgecolor='none')
plt.show()
print('✅ Static report saved as apple_report.png')

### Part C — [Matplotlib] Annual Summary Bar Chart

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.patch.set_facecolor(BG)
fig.suptitle('APPLE INC. — ANNUAL FINANCIAL SUMMARY (FY2022–2025)',
             fontsize=15, color=GOLD, fontweight='bold', y=1.02, fontfamily='serif')

fy_labels = annual.index.astype(str).tolist()
pal = [BLUE, TEAL, PURPLE, GOLD]

def saxa(ax, title):
    ax.set_facecolor('#111318')
    ax.set_title(title, color='#9AA0B0', fontsize=10, pad=8, fontfamily='monospace', fontweight='bold')
    ax.tick_params(colors='#5A6070', labelsize=9)
    ax.grid(axis='y', color=BORDER, alpha=0.9)
    ax.spines[:].set_edgecolor(BORDER)

# Annual Revenue
saxa(axes[0], 'ANNUAL REVENUE (USD Bn)')
bars = axes[0].bar(fy_labels, annual['Annual_Revenue'].values, color=pal[:len(fy_labels)], alpha=0.85, width=0.55)
for bar, val in zip(bars, annual['Annual_Revenue'].values):
    axes[0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+1,
                 f'${val:.0f}B', ha='center', fontsize=10, color='#E8EAF0', fontfamily='monospace', fontweight='bold')

# Annual Net Income
saxa(axes[1], 'ANNUAL NET INCOME (USD Bn)')
bars2 = axes[1].bar(fy_labels, annual['Annual_NetIncome'].values, color=pal[:len(fy_labels)], alpha=0.85, width=0.55)
for bar, val in zip(bars2, annual['Annual_NetIncome'].values):
    axes[1].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.3,
                 f'${val:.0f}B', ha='center', fontsize=10, color='#E8EAF0', fontfamily='monospace', fontweight='bold')

# Avg Margins
saxa(axes[2], 'AVERAGE MARGINS BY YEAR (%)')
x = np.arange(len(fy_labels)); w = 0.25
axes[2].bar(x-w, annual['Avg_Gross_Margin'].values, w, color=GREEN,  alpha=0.85, label='Gross%')
axes[2].bar(x,   annual['Avg_Op_Margin'].values,    w, color=GOLD,   alpha=0.85, label='Op%')
axes[2].bar(x+w, annual['Avg_Net_Margin'].values,   w, color=PURPLE, alpha=0.85, label='Net%')
axes[2].set_xticks(x); axes[2].set_xticklabels(fy_labels)
axes[2].legend(fontsize=8, facecolor='#111318', edgecolor=BORDER, labelcolor='#E8EAF0')

plt.tight_layout(pad=2.0)
plt.savefig('apple_annual_summary.png', dpi=150, bbox_inches='tight', facecolor=BG, edgecolor='none')
plt.show()
print('✅ Annual summary chart saved')

---
## 🏁 Section 6 — Summary & Insights

In [ ]:
latest = df_income.iloc[-1]
latest_price = df_price['Close'].iloc[-1]
total_return = df_price['Cumulative_Return'].iloc[-1] * 100

print('=' * 65)
print('  APPLE INC. (AAPL) — FINANCIAL SNAPSHOT')
print('=' * 65)
print(f'  Latest Quarter     : {latest["Quarter"]}')
print(f'  Revenue            : ${latest["Total Revenue"]:.2f}B')
print(f'  Gross Margin       : {latest["Gross_Margin"]:.1f}%')
print(f'  Operating Margin   : {latest["Op_Margin"]:.1f}%')
print(f'  Net Margin         : {latest["Net_Margin"]:.1f}%')
if 'ROE' in df_kpi.columns:
    print(f'  ROE                : {df_kpi["ROE"].iloc[-1]:.1f}%')
if 'ROA' in df_kpi.columns:
    print(f'  ROA                : {df_kpi["ROA"].iloc[-1]:.1f}%')
if 'Free Cash Flow' in df_cf.columns:
    print(f'  Latest FCF         : ${df_cf["Free Cash Flow"].iloc[-1]:.2f}B')
print(f'  Latest Stock Price : ${latest_price:.2f}')
print(f'  Total Return (period): {total_return:.1f}%')
print('=' * 65)
print()
print('📁 Output files:')
print('   apple_financial.db       — SQLite database')
print('   apple_report.png         — Matplotlib 7-panel report')
print('   apple_annual_summary.png — Annual comparison chart')
print()
print('🎉 Analysis complete!')